# wiki_search: quick start

`WikiQA` answers questions, checks claims and writes short research briefs using only English Wikipedia, with Claude driving the searches and reads. Every fact is cited to the Wikipedia section it came from.

## Setup (once)

```bash
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name wiki-search --display-name "wiki_search (.venv)"
```

Create a `.env` file in the project root:

```
ANTHROPIC_API_KEY=sk-ant-...
WIKI_USER_AGENT=wiki-search/0.1 (your-contact@example.com)
```

Wikimedia asks API clients to identify themselves; with a contact address you get throttled far less. Then open this notebook with the **wiki_search (.venv)** kernel and run the cells in order. Each question takes roughly 10 seconds to 2 minutes, and a few cents of API usage.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wiki_search").is_dir())
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from wiki_search import WikiQA

qa = WikiQA()  # Claude Sonnet 4.6; see the README for options


def show(answer):
    """Display an answer or brief, or the clarifying question if the tool asked one."""
    if answer.needs_clarification:
        c = answer.clarification
        display(Markdown(f"**The tool asks:** {c.question}\n\nOptions: {', '.join(c.options)}"))
    else:
        display(Markdown(answer.text))
    for w in answer.warnings:
        print("⚠️", w)
    print(f"({answer.elapsed_s}s)")

## A fact question

The answer comes first, then the cited evidence and sources.

In [2]:
show(qa.ask("How tall is Aconcagua?"))

**Answer:** **6,967.15 metres (22,858.1 feet)**

**Reasoning:**
- Aconcagua, located in Mendoza Province, Argentina, has a summit elevation of 6,967.15 metres (22,858.1 feet), making it the highest mountain outside Asia. [1]

**Sources:**  
[1] Aconcagua (lead) — https://en.wikipedia.org/wiki/Aconcagua

(9.1s)


## A multi-hop question

When one answer is needed to find the next, the tool resolves the chain hop by hop and cites each step: here, which country the islands belong to, then that country's currency.

In [3]:
show(qa.ask("What currency is used in the country the Galápagos Islands belong to?"))

**Answer:** **The United States dollar (USD)**

**Reasoning:**
1. The Galápagos Islands form the Galápagos Province of the Republic of Ecuador. [1]
2. The present currency of Ecuador is the United States dollar; it also mints its own centavo coins. [2]

**Sources:**  
[1] Galápagos Islands (lead) — https://en.wikipedia.org/wiki/Gal%C3%A1pagos_Islands  
[2] Currency of Ecuador (lead) — https://en.wikipedia.org/wiki/Currency_of_Ecuador

(9.6s)


## A research brief

For overviews, histories and comparisons the tool reads several articles and writes a short brief: an overview, sections, how items were chosen, further reading and sources. Before a brief is accepted, each claim is checked against the text of the section it cites; claims that misstate their source go back to the agent once to fix, and any still flagged after that are printed as warnings below the brief. This takes a minute or two.

In [2]:
show(qa.ask("Give me an overview of how the printing press changed Europe."))

## How the Printing Press Changed Europe

Invented around 1440 by Johannes Gutenberg, the printing press triggered a communications revolution that touched nearly every aspect of European life over the following two centuries. [1] By enabling the fast, cheap reproduction of identical texts, it accelerated the Reformation, sped up the Scientific Revolution, raised literacy, shaped national languages, and gave both rulers and rebels a powerful new political tool. [1][2]

### Invention and Rapid Spread
- Gutenberg combined movable type cast from a lead-tin-antimony alloy with oil-based ink and a wooden screw press, enabling a much faster rate of printing than what existed before. [2]
- From a single shop in Mainz, printing spread so that by 1480 printers were active in 110 locations across Germany, Italy, France, Spain, England, and beyond; by 1500 presses had produced over 20 million copies, growing to 150–200 million by 1600. [4]

### Literacy, Reading, and Authorship
- Lower costs and wider availability drove a shift from communal oral reading toward silent, private reading — what Eisenstein calls a transition 'from a hearing public to a reading public' — and adult literacy rose across Europe in the following two centuries. [5]
- Printing enabled precise citation, numbered pages, and indexes; it also strengthened the association between authors and their texts, and the first copyright laws were passed. [5]

### The Reformation
- Martin Luther became Europe's most published author around 1520: printers distributed roughly 300,000 copies of his works between 1518 and 1520, surprising both the pope and secular rulers. [4]
- Economic research finds that cities with a press by 1500 were 52 percentage points more likely to have adopted Protestantism by 1530, with the effect strongest in the Reformation's critical early years. [6]

### Science
- The press let natural philosophers circulate discoveries to more people across wider geographies, and enabled communities of scientists to share methods and cumulative data across generations. [7]
- Scientific paradigms were overturned faster: Copernicus's De revolutionibus (1543) supplanted Ptolemy's 1,400-year-old Almagest in under a century, a pace Eisenstein attributes largely to print rather than individual genius. [7]

### Politics and Censorship
- According to Eisenstein, citing historian Geoffrey Elton, Thomas Cromwell's 1530s campaign supporting Henry VIII's break with Rome was the first European government effort to exploit the press for propaganda; cities with early presses also grew around 60 percent faster economically than those without. [6]
- Authorities responded with censorship — the first Papal Index of Forbidden Books appeared in 1559 — but printed material crossed borders easily and clandestine presses thrived, shaping European politics from the French Wars of Religion to the French Revolution. [6]

### Language and Identity
- Books increasingly printed in local vernaculars rather than Latin helped standardize spelling and grammar region by region; Febvre and Martin argue printing did more to shape national languages than any other single factor, feeding the rise of European nationalism. [8]

**How these were chosen:** Subtopics follow the Wikipedia 'Printing press' article's own Impact section headings (authorship and reading, science, politics and church, language), supplemented by its spread section and the Gutenberg lead and Legacy section, which together represent the main areas scholars identify as transformed by the press. [1][4][5][6][7][8]

**Further reading:**
- [Protestant Reformation](https://en.wikipedia.org/wiki/Protestant_Reformation) — Explores how Luther and others exploited the press to challenge the Catholic Church across Europe.
- [Scientific Revolution](https://en.wikipedia.org/wiki/Scientific_Revolution) — Details how wider circulation of ideas helped overturn centuries of received wisdom in astronomy, medicine, and physics.
- [Index Librorum Prohibitorum](https://en.wikipedia.org/wiki/Index_Librorum_Prohibitorum) — The Church's official list of banned books — a direct response to printing and a window into the censorship struggle.
- [Gutenberg Bible](https://en.wikipedia.org/wiki/Gutenberg_Bible) — The first major book printed with movable type, a case study in the new technology's craft and ambition.
- [History of printing in Europe](https://en.wikipedia.org/wiki/History_of_printing_in_Europe) — A deeper look at how printing technology evolved and spread city by city across the continent.

**Sources:**  
[1] Printing press (lead) — https://en.wikipedia.org/wiki/Printing_press  
[2] Johannes Gutenberg (lead) — https://en.wikipedia.org/wiki/Johannes_Gutenberg  
[3] Johannes Gutenberg § Legacy — https://en.wikipedia.org/wiki/Johannes_Gutenberg#Legacy  
[4] Printing press § Mass production and spread in Europe — https://en.wikipedia.org/wiki/Printing_press#Mass_production_and_spread_in_Europe  
[5] Printing press § Authorship and reading — https://en.wikipedia.org/wiki/Printing_press#Authorship_and_reading  
[6] Printing press § Politics, state and church — https://en.wikipedia.org/wiki/Printing_press#Politics,_state_and_church  
[7] Printing press § Science — https://en.wikipedia.org/wiki/Printing_press#Science  
[8] Printing press § Language — https://en.wikipedia.org/wiki/Printing_press#Language

⚠️ Claim not backed by its citation (distorted): "Scientific paradigms were overturned faster: Copernicus's De revolutionibus (1543) supplanted Ptolemy's 1,400-year-old Almagest in under a century, a pace Eisenstein attributes largely to print rather than individual genius.". The cited text: Ptolemy's Almagest, published in the 2nd century AD, remained authoritative as an astronomical text for some 1400 years before Copernicus's De revolutionibus surpassed it in 1543, but De revolutionibus was surpassed in less than 100 years.
(118.9s)


## Ambiguous names: when the tool asks, and when it doesn't

If a question could reasonably mean several things, the tool asks which one you meant; call `ask` again with the reply. It doesn't ask when the wording or common usage makes one meaning clear.

**Asks:** there are many Springfields, and nothing in the question says which one.

In [5]:
question = "What is the population of Springfield?"
first = qa.ask(question)
show(first)
if first.needs_clarification:
    show(qa.ask(question, clarification='By "Springfield" I mean Springfield, Illinois.'))

**The tool asks:** There are many places named Springfield! Could you clarify which Springfield you're asking about? For example, are you referring to Springfield, Missouri (the most populous city with that name), Springfield, Illinois (the state capital), Springfield, Massachusetts, Springfield, Ohio, or another Springfield entirely?

Options: Springfield, Missouri, Springfield, Illinois, Springfield, Massachusetts, Springfield, Ohio, Springfield, Oregon, Springfield, Virginia

(8.2s)


**Answer:** **114,394 (2020 U.S. Census); ~208,000 in the metro area**

**Reasoning:**
- Springfield, Illinois had a population of 114,394 at the 2020 United States Census, making it the state's seventh-most populous city. [1]
- Approximately 208,000 residents live in the Springfield metropolitan area, which consists of all of Sangamon and Menard counties. [1]

**Sources:**  
[1] Springfield, Illinois (lead) — https://en.wikipedia.org/wiki/Springfield,_Illinois

(8.3s)


**Doesn't ask:** "Macbeth" is also an opera and a Scottish king, but the play is the obvious reading. The tool answers for it and says which meaning it took.

In [6]:
show(qa.ask("When was Macbeth written?"))

**Answer:** **Around 1606 (first performed 1606; first published 1623)**

**Reasoning:**
- Macbeth is estimated to have been first performed in 1606 and was first published in the Folio of 1623. [1]

**Sources:**  
[1] Macbeth (lead) — https://en.wikipedia.org/wiki/Macbeth

(7.8s)
